# DTU dataset, clean speech: the DTU ROI (sensitivity analysis B)

The single-talker trials of the DTU dataset (Fuglsang, Dau & Hjortkjaer 2017), analysed as this project's `clean.ipynb`: does adding the onset spectrogram, or the envelope spectrogram, improve the prediction of held-out EEG (one-tailed TFCE over the whole scalp)? The DTU ROI is the sensors where both do (the conjunction, this project's rule). It is used for **sensitivity analysis B** in the DTU two-talker notebooks; their **primary** analysis uses this project's own clean-speech ROI (`roi.txt`), pre-specified in an independent sample. DTU's single-talker trials are ten repetitions of one 50 s speech clip (not this project's clean-speech condition), which is why this ROI is a sensitivity analysis rather than the primary one.

**How the data were prepared** (`dtu/dtu_pipeline.py`, on the cluster; run from `dtu.ipynb`), aligned with this project step for step:
- EEG: the raw recordings, preprocessed as this project's pipeline: bad channels by neighbour correlation, 0.5-20 Hz, mastoid reference, extended-infomax ICA with ICLabel (eye blinks, heart beats at >= 0.9), 64 Hz; trials cut at the trigger events as the authors' script does
- Trials: from the authors' trial table (stories, attended talker, single or two talkers, room)
- Predictors: 8-band gammatone envelope and onsets, made exactly as this project's; the mixture from the two stories summed (the rooms were simulated at presentation, so the exact mixture at the ears is not available)
- TRFs: the same boosting settings (-100 to 400 ms, 50 ms basis, l1, cross-validated)
- Tests: as this project's two-talker notebooks (one-tailed TFCE in the ROI, and the ROI-averaged t-test); repeated at the end in this project's own ROI (`roi.txt`) as a check

**Results come from the cluster**: run `dtu_run()` in `dtu.ipynb` first; this notebook only reads them.

In [ ]:
# Reloads the code every time it runs, so changes pulled from GitHub take effect without
# restarting the kernel (Run All is enough).
import importlib, sys
for _module in ('experiment', 'parameters'):
    if getattr(sys.modules.get(_module), '__spec__', None):
        importlib.reload(sys.modules[_module])
from parameters import *

In [ ]:
# The clean-speech results (whole-scalp tests) and the ROIs
ROI = dtu_setup('clean')
ROI, _ = dtu_rois('clean')  # the DTU ROI, for the ROI-averaged tests in this notebook
configure_analysis(roi=ROI)

In [ ]:
# The full clean-speech model's TRFs (attended envelope and onsets)
data_full = dtu_trfs('clean')

<small>

**Onset (DTU clean speech), test - baseline**: does adding the onset spectrogram improve the prediction?

</small>

In [ ]:
data, diff, result = dtu_model_test('Onset', 'clean')
show_topomap_trio(data, diff, result, 'Onset', 'DTU clean', None)

In [ ]:
display(result)
print(f"Onset (DTU clean): TFCE {tfce_summary(result)}")
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset (DTU clean): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

<small>

**Envelope (DTU clean speech), test - baseline**: does adding the envelope spectrogram improve the prediction?

</small>

In [ ]:
data, diff, result = dtu_model_test('Envelope', 'clean')
show_topomap_trio(data, diff, result, 'Envelope', 'DTU clean', None)

In [ ]:
display(result)
print(f"Envelope (DTU clean): TFCE {tfce_summary(result)}")
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope (DTU clean): ROI model comparison t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

## The DTU ROI (sensitivity B)

The sensors where both tests above are significant (or, if none, those significant for either; the rule used is printed above), next to this project's ROI, which the primary analysis uses.

In [ ]:
p = plot.SensorMap(data_full['gammatone_on_8'].sensor, mark=ROI, title='DTU ROI (DTU clean speech)')
_, cocktail = dtu_rois('clean')
p = plot.SensorMap(data_full['gammatone_on_8'].sensor, mark=cocktail, title="This project's ROI (roi.txt)")
print(f"In both: {len(set(ROI) & set(cocktail))} sensors; DTU only: {sorted(set(ROI) - set(cocktail))}; this project only: {sorted(set(cocktail) - set(ROI))}")

## Clean speech response functions

In [ ]:
envelope_strf = extract_strf(data_full, 'gammatone_8')
onset_strf = extract_strf(data_full, 'gammatone_on_8')
p = plot.Array([envelope_strf * 1e3, onset_strf * 1e3], axtitle=['Envelope', 'Onset'],
               title='Clean speech STRF (DTU)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

In [ ]:
data = extract_trf(data_full, 'gammatone_on_8')
data = add_anchored_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset (DTU clean) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset (DTU clean) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")
p = plot_trf_bins(data_full, 'gammatone_on_8', 'Onset TRF (DTU clean)')

In [ ]:
data = extract_trf(data_full, 'gammatone_8')
data = add_anchored_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope (DTU clean) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope (DTU clean) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")
p = plot_trf_bins(data_full, 'gammatone_8', 'Envelope TRF (DTU clean)')